In [3]:
# ============================================================
# Carbon-source-specific FBAwMC
# ============================================================

from cobra.io import read_sbml_model
import numpy as np
import pandas as pd

model = read_sbml_model("../models/iJR904.xml.gz")

# ------------------------------------------------------------
# Carbon sources
# ------------------------------------------------------------

carbon_sources = {
    "Glucose": "EX_glc__D_e",
    "Maltose": "EX_malt_e",
    "Galactose": "EX_gal_e",
    "Glycerol": "EX_glyc_e",
    "Lactate": "EX_lac__L_e",
    "Acetate": "EX_ac_e"
}


# ------------------------------------------------------------
# Carbon-source-specific average crowding coefficients
# Reported by Beg et al.
# ------------------------------------------------------------

mean_a_by_carbon_source = {
    "Glucose": 0.0031,
    "Glycerol": 0.0053,
    "Maltose": 0.0040,
    "Galactose": 0.0040,
    "Lactate": 0.0040,
    "Acetate": 0.0040
}


# ------------------------------------------------------------
# Base medium
# ------------------------------------------------------------

base_medium = model.medium.copy()

# Remove all six carbon sources from the base medium
for rxn_id in carbon_sources.values():
    base_medium.pop(rxn_id, None)

# Oxygen in excess
base_medium["EX_o2_e"] = 999999.0


# ------------------------------------------------------------
# Reactions subject to molecular crowding
# ------------------------------------------------------------

crowding_reactions = [
    rxn for rxn in model.reactions
    if rxn not in model.exchanges
    and "BIOMASS" not in rxn.id.upper()
]


# ------------------------------------------------------------
# Molecular crowding constraint
# ------------------------------------------------------------

crowding_constraint = model.problem.Constraint(
    0,
    ub=1.0,
    name="molecular_crowding"
)

model.add_cons_vars(crowding_constraint)
model.solver.update()


# ------------------------------------------------------------
# Simulation parameters
# ------------------------------------------------------------

n_runs = 1000
beta = 3

# Fixed seed for reproducibility
rng = np.random.default_rng(seed=42)


# ------------------------------------------------------------
# Run simulations
# ------------------------------------------------------------

results_fbawmc_specific = []


for run in range(n_runs):

    for name, exchange_id in carbon_sources.items():

        # Get carbon-source-specific average crowding coefficient
        mean_a = mean_a_by_carbon_source[name]

        # ----------------------------------------------------
        # Generate reaction-specific crowding coefficients
        # from Gamma distribution
        # ----------------------------------------------------

        a_values = rng.gamma(
            shape=beta,
            scale=mean_a / beta,
            size=len(crowding_reactions)
        )

        # ----------------------------------------------------
        # Assign coefficients to molecular crowding constraint
        # ----------------------------------------------------

        coefficients = {}

        for rxn, a_i in zip(crowding_reactions, a_values):

            coefficients[rxn.forward_variable] = a_i
            coefficients[rxn.reverse_variable] = a_i

        crowding_constraint.set_linear_coefficients(coefficients)

        # ----------------------------------------------------
        # Define medium for current carbon source
        # ----------------------------------------------------

        medium = base_medium.copy()
        uptake_bounds = {
            "Glucose": 10,
            "Galactose": 10,
            "Maltose": 5,
            "Glycerol": 20,
            "Lactate": 20,
            "Acetate": 30
        }

        medium[exchange_id] = uptake_bounds[name]
        model.medium = medium

        # ----------------------------------------------------
        # Optimize FBAwMC model
        # ----------------------------------------------------

        solution = model.optimize()

        if solution.status == "optimal":

            growth = solution.objective_value
            uptake = abs(solution.fluxes[exchange_id])

        else:

            growth = np.nan
            uptake = np.nan

        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        results_fbawmc_specific.append({
            "run": run + 1,
            "carbon_source": name,
            "mean_a": mean_a,
            "growth_rate": growth,
            "uptake_rate": uptake
        })


# ------------------------------------------------------------
# Convert results to DataFrame
# ------------------------------------------------------------

df_fbawmc_specific = pd.DataFrame(
    results_fbawmc_specific
)


# Display first results
df_fbawmc_specific.head(10)

OSError: The file with '../models/iJR904.xml.gz' does not exist, or is not an SBML string. Provide the path to an existing SBML file or a valid SBML string representation:


In [ ]:
# ============================================================
# Summary of carbon-source-specific FBAwMC results
# ============================================================

summary_fbawmc_specific = (
    df_fbawmc_specific
    .groupby("carbon_source")
    .agg(
        mean_growth=("growth_rate", "mean"),
        std_growth=("growth_rate", "std"),
        mean_uptake=("uptake_rate", "mean"),
        mean_a=("mean_a", "first")
    )
)

summary_fbawmc_specific